# AfriVoices — VALIDATION MATÉRIELLE de la soumission (règlement : 1 rapport / soumission)

Mesure, **sur CPU (4 threads, régime edge)** et avec la **config exacte de la soumission**
(modèle + KenLM + (α, β) + troncature + décodage direct) :

1. **RAM** : RSS après chargement du modèle (fp32) + pire KenLM (swa), méthode documentée ;
2. **RTF par clip** : échantillon stratifié (langue × type, clips courts ET longs) —
   moyenne / p50 / p95 / max ;
3. **Courbe pic-mémoire = f(durée du clip)** : le transitoire d'attention croît en T² —
   mesure du seuil où le décodage direct dépasse 8 Go, et du pic du **repli fenêtré**
   intégré au générateur (qui garantit la tenue mémoire au-delà du seuil) ;
4. **Latence projetée sur l'INTÉGRALITÉ du test** : durée audio totale (estimée sur
   échantillon, ou exacte avec `EXACT_DUREE=True`) × RTF mesuré ;
5. Verdicts de conformité (params ≤ 1 Md, RAM ≤ 8 Go, RTF ≤ 2×) ;
6. Écrit un rapport **daté** `validation/HARDWARE_VALIDATION_{TAG}_{date}.md` sur le
   Drive — à committer dans le dépôt, un par soumission sélectionnée.

**Session CPU FRAÎCHE (runtime redémarré)** — indispensable : le pic mémoire se mesure sans bagage de session. Durée : ~30-50 min avec
`N_PAR_CASE=10` (les clips spontanés longs dominent le temps — c'est voulu, ils bornent
le RTF max). Honnêteté : le rapport porte sa date de mesure ; pour une soumission déjà
envoyée, il documente *le même pipeline, mesuré a posteriori*.

## 1. Dépendances (redémarrer le runtime après la 1ère exécution)

In [ ]:
import importlib, subprocess, sys
need = [m for m in ["kenlm", "pyctcdecode", "psutil"] if importlib.util.find_spec(m) is None]
if need:
    print("installation de", need, "...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "pyctcdecode", "pygtrie"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "psutil"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "https://github.com/kpu/kenlm/archive/master.zip"], check=False)
    print("⚠️ REDÉMARRE le runtime, puis relance cette cellule.")
else:
    import numpy
    print(f"✓ modules présents (numpy {numpy.__version__}) — continue en §2")

## 2. CONFIG — doit être IDENTIQUE à la soumission validée

In [ ]:
# ============================================================
MODEL_NAME = "baobab-asr-v9-2"
LM_SUBDIR  = "lm_v2"
ALPHA, BETA = 0.5, 0.0              # config FINALE (0.39477). Contrôle 0.39923 : (0.7, 0.5)
UNIGRAM_TOP = 50000
TAG = "v9_2_polB_0.39477"           # étiquette de la soumission validée
THREADS = 4                         # régime edge de référence
N_PAR_CASE = 10                     # clips par (langue × type) pour le RTF
N_DUREE = 600                       # clips pour estimer la durée audio totale du test
EXACT_DUREE = False                 # True = scan complet des 41 733 durées (~45-90 min en plus)
# ============================================================

from google.colab import drive
drive.mount("/content/drive")
import os, torch
BASE  = "/content/drive/MyDrive/afrivoices"
MODEL = f"{BASE}/{MODEL_NAME}"
LM    = f"{BASE}/{LM_SUBDIR}"
OUTD  = f"{BASE}/validation"
os.makedirs(OUTD, exist_ok=True)
assert os.path.isdir(MODEL) and os.path.isdir(LM)
torch.set_num_threads(THREADS)
print(f"config: {MODEL_NAME} + {LM_SUBDIR} (α={ALPHA}, β={BETA}) | CPU {THREADS} threads | tag {TAG}")

## 3. Matériel + RAM (méthode : RSS via psutil, avant/après chargements)

In [ ]:
import psutil, platform, time, io, base64, re, numpy as np, soundfile as sf, librosa
from collections import Counter

proc = psutil.Process()
def rss_gb(): return proc.memory_info().rss / 1e9

cpu_info = platform.processor() or platform.machine()
try:
    cpu_info = [l.split(":")[1].strip() for l in open("/proc/cpuinfo") if "model name" in l][0]
except Exception:
    pass
HW = {"cpu": cpu_info, "coeurs_logiques": psutil.cpu_count(), "threads_utilises": THREADS,
      "ram_totale_gb": round(psutil.virtual_memory().total / 1e9, 1),
      "os": platform.platform()}
print(HW)

rss0 = rss_gb()
assert rss0 < 2.0, (f"SESSION NON FRAÎCHE : RSS de base = {rss0:.2f} Go (attendu < 0.5). "
                    "Exécution -> Déconnecter et supprimer l'environnement, puis relance §1->§6. "
                    "Toute mesure dans cette session serait polluée.")
t0 = time.perf_counter()
from transformers import Wav2Vec2BertForCTC, Wav2Vec2BertProcessor
model = Wav2Vec2BertForCTC.from_pretrained(MODEL, local_files_only=True).eval()
processor = Wav2Vec2BertProcessor.from_pretrained(MODEL, local_files_only=True)
t_load_model = time.perf_counter() - t0
rss_model = rss_gb()
assert rss_model > rss0, f"RSS décroissant ({rss0:.2f} -> {rss_model:.2f}) : session polluée, redémarre proprement."

raw = [t for t, _ in sorted(processor.tokenizer.get_vocab().items(), key=lambda kv: kv[1])]
blank_tok = processor.tokenizer.pad_token
labels, n_ = [], 0
for t in raw:
    if t == blank_tok: labels.append("")
    elif t in ("|", " "): labels.append(" ")
    elif t in {"[UNK]", "<s>", "</s>"}: n_ += 1; labels.append("\u2047" * n_)
    else: labels.append(t)

def unigrams(lang, top=UNIGRAM_TOP):
    c = Counter()
    with open(f"{LM}/{lang}.txt", encoding="utf-8") as f:
        for line in f: c.update(line.split())
    return [w for w, _ in c.most_common(top)]

from pyctcdecode import build_ctcdecoder
t0 = time.perf_counter()
dec_swa = build_ctcdecoder(labels, kenlm_model_path=f"{LM}/swa.bin",
                           unigrams=unigrams("swa"), alpha=ALPHA, beta=BETA)
t_load_lm = time.perf_counter() - t0
rss_full = rss_gb()

n_params = sum(p.numel() for p in model.parameters()) / 1e9
print(f"\nparams: {n_params:.3f} Md")
print(f"RSS: base {rss0:.2f} | +modèle {rss_model:.2f} | +KenLM swa (pire cas) {rss_full:.2f} Go")
print(f"chargements: modèle {t_load_model:.0f}s | décodeur swa {t_load_lm:.0f}s")

decoders = {"swa": dec_swa}
def dec_de(lang):
    if lang not in decoders:
        decoders[lang] = build_ctcdecoder(labels, kenlm_model_path=f"{LM}/{lang}.bin",
                                          unigrams=unigrams(lang), alpha=ALPHA, beta=BETA)
    return decoders[lang]

def decode_bytes(b):
    try: arr, sr = sf.read(io.BytesIO(b), dtype="float32")
    except Exception: arr, sr = sf.read(io.BytesIO(base64.b64decode(b)), dtype="float32")
    if arr.ndim > 1: arr = arr.mean(axis=1)
    if sr != 16000: arr = librosa.resample(arr, orig_sr=sr, target_sr=16000)
    return arr.astype(np.float32)

def duree_bytes(b):
    try: i = sf.info(io.BytesIO(b)); return i.frames / i.samplerate
    except Exception:
        try: i = sf.info(io.BytesIO(base64.b64decode(b))); return i.frames / i.samplerate
        except Exception: return None

### 3bis — Traqueur de pic mémoire (le RSS se lit PENDANT, pas après)

In [ ]:
import threading, gc

class PeakRSS:
    def __init__(self, interval=0.05):
        self.interval = interval
        self.peak = proc.memory_info().rss / 1e9
        self._stop = False
    def _run(self):
        while not self._stop:
            self.peak = max(self.peak, proc.memory_info().rss / 1e9)
            time.sleep(self.interval)
    def __enter__(self):
        self._stop = False
        self.t = threading.Thread(target=self._run, daemon=True)
        self.t.start(); return self
    def __exit__(self, *a):
        self._stop = True; self.t.join(timeout=1)

print("traqueur prêt (échantillonnage RSS à 50 ms)")

## 4. Échantillons du test : RTF stratifié (langue × type) + durée totale

In [ ]:
import glob, pandas as pd, random
tp = sorted(glob.glob(f"{BASE}/test/**/*.parquet", recursive=True)) or \
     sorted(glob.glob("/content/test_data/**/*.parquet", recursive=True))
if not tp:
    from huggingface_hub import snapshot_download
    snapshot_download("digitalumuganda/anv-test-data-nt", repo_type="dataset", local_dir="/content/test_data")
    tp = sorted(glob.glob("/content/test_data/**/*.parquet", recursive=True))
print(len(tp), "parquets test")

rng = random.Random(42)
# --- échantillon RTF : N_PAR_CASE par (langue, type visible dans le nom de fichier) ---
sample = []   # (lang, typ, bytes)
quota = {}
files = tp[:]; rng.shuffle(files)
for p in files:
    typ = "unscripted" if "unscripted" in os.path.basename(p).lower() else "scripted"
    try: df = pd.read_parquet(p)
    except Exception: continue
    for _, r in df.sample(frac=1, random_state=42).iterrows():
        lang = r.get("language")
        key = (lang, typ)
        if quota.get(key, 0) >= N_PAR_CASE: continue
        b = r["audio"].get("bytes") if isinstance(r["audio"], dict) else r["audio"]
        if not b: continue
        sample.append((lang, typ, b)); quota[key] = quota.get(key, 0) + 1
    if len(quota) >= 11 and all(v >= N_PAR_CASE for v in quota.values()):
        break
print("échantillon RTF:", {k: v for k, v in sorted(quota.items())}, "| total", len(sample))

# --- durée audio totale du test ---
if EXACT_DUREE:
    durs = []
    tot_s, n_tot = 0.0, 0
    for p in tp:
        df = pd.read_parquet(p)
        for _, r in df.iterrows():
            b = r["audio"].get("bytes") if isinstance(r["audio"], dict) else r["audio"]
            d = duree_bytes(b) if b else None
            if d: tot_s += d; n_tot += 1; durs.append(d)
    duree_totale_h = tot_s / 3600
    methode_duree = f"scan exact ({n_tot} clips)"
else:
    durs = []
    for p in files:
        try: df = pd.read_parquet(p)
        except Exception: continue
        for _, r in df.sample(frac=1, random_state=7).head(12).iterrows():
            b = r["audio"].get("bytes") if isinstance(r["audio"], dict) else r["audio"]
            d = duree_bytes(b) if b else None
            if d: durs.append(d)
        if len(durs) >= N_DUREE: break
    m, s = float(np.mean(durs)), float(np.std(durs))
    duree_totale_h = m * 41733 / 3600
    ic = 1.96 * s / np.sqrt(len(durs)) * 41733 / 3600
    methode_duree = f"estimation sur {len(durs)} clips (moy {m:.1f}s ± IC95 {ic:.1f} h sur le total)"
print(f"durée audio totale du test ≈ {duree_totale_h:.1f} h ({methode_duree})")

## 5. Mesure RTF (pipeline complet : features + forward CPU + décodage LM)

In [ ]:
import time
res = []   # (lang, typ, durée, temps, delta_pic)
for i, (lang, typ, b) in enumerate(sample):
    try:
        arr = decode_bytes(b)
    except Exception:
        continue
    dur = len(arr) / 16000.0
    dec = dec_de(lang)
    gc.collect()
    r0 = rss_gb()
    with PeakRSS() as mon:
        t0 = time.perf_counter()
        fe = processor.feature_extractor([arr], sampling_rate=16000, return_tensors="pt")
        with torch.no_grad():
            lg = model(**fe).logits[0].numpy()
        _ = dec.decode(lg)
        dt = time.perf_counter() - t0
    res.append((lang, typ, dur, dt, mon.peak - r0))
    del arr, lg, fe
    if (i + 1) % 10 == 0:
        print(f"  {i+1}/{len(sample)} clips (RSS {rss_gb():.2f} Go)")

import numpy as np
rtf = np.array([t / d for _, _, d, t, _ in res])
print(f"\nRTF : moy {rtf.mean():.3f} | p50 {np.median(rtf):.3f} | p95 {np.percentile(rtf,95):.3f} | max {rtf.max():.3f}")
print(f"{'case':20} {'n':>3} {'dur.moy':>8} {'RTF moy':>8} {'RTF max':>8} {'Dpic max':>9}")
cases = sorted(set((l, t) for l, t, _, _, _ in res))
for (l, t) in cases:
    sel = [(d, dt, pk) for l2, t2, d, dt, pk in res if (l2, t2) == (l, t)]
    r = np.array([dt / d for d, dt, _ in sel])
    print(f"{l}/{t:12} {len(sel):>3} {np.mean([d for d,_,_ in sel]):>7.1f}s {r.mean():>8.3f} {r.max():>8.3f} {max(pk for _,_,pk in sel):>8.2f}G")

latence_totale_h = duree_totale_h * float(rtf.mean())
print(f"\nLatence projetée sur TOUT le test : {duree_totale_h:.1f} h audio x RTF {rtf.mean():.3f} = {latence_totale_h:.1f} h (CPU {THREADS} threads)")

## 5bis. Courbe pic-mémoire = f(durée) + pic du REPLI FENÊTRÉ

Le transitoire d'attention croît en T² : on tranche le clip le plus long de
l'échantillon à 10/20/30/45/60/90 s et on mesure le **delta de pic** de chaque passe.
Pic « processus propre » estimé = RSS(modèle + 1 décodeur, mesuré en §3 avant toute
inférence) + delta. Puis mesure du pic avec le repli fenêtré du générateur
(fenêtres ≤ 16 s + recollage des logits) sur le clip complet.

In [ ]:
durs_ech = []
for lang, typ, b in sample:
    d = duree_bytes(b)
    durs_ech.append((d or 0, lang, typ, b))
d_max, lang0, typ0, b0 = max(durs_ech)
arr_full = decode_bytes(b0)
dec0 = dec_de(lang0)
print(f"clip de référence : {lang0}/{typ0}, {d_max:.0f}s")

steady = rss_full   # §3 : RSS après modèle + décodeur swa, AVANT toute inférence
mem_rows = []
for dsec in [10, 20, 30, 45, 60, 90]:
    n = int(min(dsec, d_max) * 16000)
    seg = arr_full[:n]
    gc.collect(); r0 = rss_gb()
    with PeakRSS() as mon:
        fe = processor.feature_extractor([seg], sampling_rate=16000, return_tensors="pt")
        with torch.no_grad():
            lg = model(**fe).logits[0].numpy()
        _ = dec0.decode(lg)
    delta = mon.peak - r0
    pic_propre = steady + delta
    mem_rows.append((min(dsec, d_max), delta, pic_propre))
    print(f"  {min(dsec, d_max):>4.0f}s : Dpic {delta:6.2f} Go | pic processus propre = {pic_propre:6.2f} Go "
          + ("OK <= 8" if pic_propre <= 8 else "DEPASSE 8"))
    del seg, lg, fe
    if dsec >= d_max: break

ok_durs = [d for d, _, p in mem_rows if p <= 8.0]
seuil_s = max(ok_durs) if ok_durs else 10.0
seuil_txt = (f"~{seuil_s:.0f}s" if ok_durs and len(ok_durs) < len(mem_rows)
             else ("toutes durées testées" if len(ok_durs) == len(mem_rows) else "< 10s"))
print(f"\nseuil de conformité 8 Go en décodage direct : {seuil_txt}")

# --- repli fenêtré (celui du générateur final) sur le clip complet ---
WIN_SEC, OVER_SEC, SEARCH_SEC = 15.0, 2.0, 3.0
def _rms(x, fl=400, hop=160):
    if len(x) < fl:
        return np.array([float(np.sqrt(np.mean(x ** 2) + 1e-12))]), hop
    nfr = 1 + (len(x) - fl) // hop
    idx = np.arange(nfr)[:, None] * hop + np.arange(fl)[None, :]
    return np.sqrt((x[idx] ** 2).mean(axis=1) + 1e-12), hop

def fenetres_v2(arr, sr=16000):
    L = len(arr); w = int(WIN_SEC * sr); o = int(OVER_SEC * sr); s_s = int(SEARCH_SEC * sr)
    if L <= w: return [(0, L)]
    wins = []; cur = 0
    while L - cur > w:
        target = cur + w
        a = max(target - s_s, cur + o)
        r, hop = _rms(arr[a:min(target, L)])
        cut = min(max(a + int(np.argmin(r)) * hop + 200, cur + o), L - 1)
        wins.append((cur, min(cut + o // 2, L)))
        cur = max(cut - o // 2, 0)
    wins.append((max(0, L - w), L))
    if len(wins) >= 2 and wins[-1][0] <= wins[-2][0]:
        wins[-2] = (wins[-2][0], L); wins.pop()
    return wins

def stitch(wins, lgs):
    if len(wins) == 1: return lgs[0].astype(np.float32)
    parts = []
    for i, ((a, e), lg) in enumerate(zip(wins, lgs)):
        nfr = lg.shape[0]
        if nfr == 0: continue
        spf = (e - a) / nfr
        lo = a if i == 0 else (wins[i - 1][1] + a) // 2
        hi = e if i == len(wins) - 1 else (e + wins[i + 1][0]) // 2
        x0 = max(0, int(round((lo - a) / spf))); x1 = min(nfr, int(round((hi - a) / spf)))
        if x1 > x0: parts.append(lg[x0:x1].astype(np.float32))
    return np.concatenate(parts, axis=0)

gc.collect(); r0 = rss_gb()
t0_fb = time.perf_counter()
with PeakRSS() as mon:
    wins = fenetres_v2(arr_full)
    lgs = []
    for (a, e) in wins:
        fe = processor.feature_extractor([arr_full[a:e]], sampling_rate=16000, return_tensors="pt")
        with torch.no_grad():
            lgs.append(model(**fe).logits[0].numpy())
    _ = dec0.decode(stitch(wins, lgs))
t_fb = time.perf_counter() - t0_fb
rtf_fb = t_fb / d_max
delta_fb = mon.peak - r0
pic_fb = steady + delta_fb
print(f"RTF du repli fenêtré sur {d_max:.0f}s : {rtf_fb:.3f}")
print(f"repli fenêtré ({len(wins)} fenêtres, clip {d_max:.0f}s) : Dpic {delta_fb:.2f} Go | pic propre = {pic_fb:.2f} Go "
      + ("OK <= 8" if pic_fb <= 8 else "DEPASSE 8"))

## 6. Rapport daté → `validation/HARDWARE_VALIDATION_{TAG}_{date}.md`

In [ ]:
import datetime
date = datetime.date.today().isoformat()
ok_params = n_params <= 1.0
ok_ram = pic_fb <= 8.0

# --- RTF par régime déployé ---
court = [(d, dt) for l, t, d, dt, _ in res if d <= seuil_s]
rtf_c = np.array([dt / d for d, dt in court])
rtf_c_agg = sum(dt for _, dt in court) / sum(d for d, _ in court)
ok_rtf = (rtf_c_agg <= 2.0) and (float(np.percentile(rtf_c, 95)) <= 2.0) and (rtf_fb <= 2.0)

# --- projection à deux régimes (pondérée par le temps d'audio) ---
durs_arr = np.array(durs, dtype=float)
p_long = float(durs_arr[durs_arr > seuil_s].sum() / durs_arr.sum())
rtf_deploye = (1 - p_long) * rtf_c_agg + p_long * rtf_fb
latence_totale_h = duree_totale_h * rtf_deploye
print(f"part du temps d'audio au-delà du seuil ({seuil_s:.0f}s) : {100*p_long:.1f}%")
print(f"RTF déployé (2 régimes) : {rtf_deploye:.3f} -> latence totale ≈ {latence_totale_h:.1f} h")

L = []
L.append(f"# Validation matérielle — soumission {TAG}")
L.append("")
L.append(f"*Mesuré le {date}, session CPU fraîche, pipeline identique à la soumission "
         f"(modèle, KenLM, (α, β)=({ALPHA}, {BETA}), troncature anti-padding, décodage direct "
         f"avec repli fenêtré intégré au-delà du seuil mémoire).*")
L.append("")
L.append("## Matériel de mesure")
L.append(f"- CPU : {HW['cpu']} ({HW['coeurs_logiques']} cœurs logiques, **{THREADS} threads utilisés** — régime edge)")
L.append(f"- RAM totale : {HW['ram_totale_gb']} Go | OS : {HW['os']}")
L.append("")
L.append("## Conformité")
L.append("")
L.append("| Critère | Limite | Mesuré | Statut |")
L.append("|---|---|---|---|")
L.append(f"| Paramètres | ≤ 1 Md | {n_params:.3f} Md | {'✓' if ok_params else '✗'} |")
L.append(f"| RAM plancher (modèle fp32 + pire KenLM, avant inférence) | ≤ 8 Go | {steady:.2f} Go | {'✓' if steady <= 8 else '✗'} |")
pic_court = max(p for d, _, p in mem_rows if d <= seuil_s)
L.append(f"| RAM pic — clips ≤ {seuil_s:.0f}s, décodage direct | ≤ 8 Go | {pic_court:.2f} Go | {'✓' if pic_court <= 8 else '✗'} |")
L.append(f"| RAM pic — clips > {seuil_s:.0f}s, **repli fenêtré** | ≤ 8 Go | {pic_fb:.2f} Go | {'✓' if ok_ram else '✗'} |")
L.append(f"| RTF — clips ≤ {seuil_s:.0f}s, direct (agrégé / p95 / max) | ≤ 2× | {rtf_c_agg:.3f} / {np.percentile(rtf_c,95):.3f} / {rtf_c.max():.3f} | {'✓' if rtf_c_agg <= 2 and np.percentile(rtf_c,95) <= 2 else '✗'} |")
L.append(f"| RTF — clips > {seuil_s:.0f}s, repli fenêtré (mesuré, {d_max:.0f}s) | ≤ 2× | {rtf_fb:.3f} | {'✓' if rtf_fb <= 2 else '✗'} |")
L.append("")
L.append("## Mémoire : pic = f(durée du clip) — décodage direct")
L.append("")
L.append(f"Le transitoire d'attention du conformer croît en T². Pic « processus propre » = plancher ({steady:.2f} Go) + delta transitoire mesuré :")
L.append("")
L.append("| durée | Δ pic transitoire | pic processus propre | ≤ 8 Go |")
L.append("|---|---|---|---|")
for d, delta, p in mem_rows:
    L.append(f"| {d:.0f}s | {delta:.2f} Go | {p:.2f} Go | {'✓' if p <= 8 else '✗'} |")
L.append("")
L.append(f"**Seuil de conformité en décodage direct : {seuil_txt}.** Au-delà, le générateur bascule "
         f"automatiquement sur le **repli fenêtré** (fenêtres ≤ 16 s + recollage des logits) : pic "
         f"{pic_fb:.2f} Go et RTF {rtf_fb:.3f} mesurés sur un clip de {d_max:.0f} s — le repli résout "
         f"simultanément la contrainte mémoire ET la contrainte de vitesse (le RTF direct d'un clip de 90 s "
         f"atteint {max(dt/d for _, _, d, dt, _ in res):.2f} : régime non déployé). Coût qualité du repli "
         f"mesuré au préalable (RAPPORT Partie II §3) : appliqué à tous les clips > 20 s il coûterait "
         f"≈ +0.007 de macro ; appliqué au-delà du seuil seulement, son impact est inférieur. Sur GPU / "
         f"machines à grande RAM, le décodage direct s'applique partout (configuration de la soumission).")
L.append("")
L.append("## Latence d'inférence sur l'intégralité du test")
L.append(f"- Durée audio totale : **{duree_totale_h:.1f} h** ({methode_duree})")
L.append(f"- Répartition du temps d'audio : {100*(1-p_long):.1f}% en clips ≤ {seuil_s:.0f}s (direct, RTF agrégé {rtf_c_agg:.3f}), "
         f"{100*p_long:.1f}% en clips > {seuil_s:.0f}s (repli, RTF {rtf_fb:.3f})")
L.append(f"- **Latence projetée : ≈ {latence_totale_h:.1f} h** sur CPU {THREADS} threads (41 733 clips, RTF déployé {rtf_deploye:.3f})")
L.append("")
L.append(f"## Détail RTF par (langue × type) — décodage direct, échantillon stratifié n={len(res)}")
L.append("| case | n | durée moy | RTF moy | RTF max |")
L.append("|---|---|---|---|---|")
for (l, t) in cases:
    sel = [(d, dt) for l2, t2, d, dt, _ in res if (l2, t2) == (l, t)]
    L.append(f"| {l}/{t} | {len(sel)} | {np.mean([d for d,_ in sel]):.1f}s | "
             f"{np.mean([dt/d for d,dt in sel]):.3f} | {max(dt/d for d,dt in sel):.3f} |")
L.append("")
L.append("## Méthode")
L.append(f"- Pic RSS échantillonné à 50 ms PENDANT chaque passe (psutil, thread dédié). Session fraîche : "
         f"plancher = base {rss0:.2f} → +modèle {rss_model:.2f} → +KenLM swa {rss_full:.2f} Go (mmap, pire cas compté).")
L.append("- RTF = temps mur (features + forward CPU + beam pyctcdecode) / durée, par clip. Le tableau détaillé "
         "documente le décodage direct sur toutes durées ; la conformité s'évalue par régime déployé "
         f"(direct ≤ {seuil_s:.0f}s, repli au-delà).")
L.append(f"- Latence totale = durée audio × RTF pondéré par régime (répartition des durées mesurée sur l'échantillon de durée).")
L.append(f"- Chargements : modèle {t_load_model:.0f} s, décodeur swa {t_load_lm:.0f} s (hors RTF).")
rapport = "\n".join(L)

path = f"{OUTD}/HARDWARE_VALIDATION_{TAG}_{date}.md"
with open(path, "w", encoding="utf-8") as f:
    f.write(rapport)
print("✅ rapport écrit ->", path)
print("\nRelance ce notebook (session fraîche) avec ALPHA,BETA=(0.7,0.5) et TAG='v9_2_controle_0.39923'.")